# Task 1: spatial autoencoder GPU training
Select **Runtime → Change runtime type → GPU** before running. Upload the provided source ZIP when prompted. Drive permission is granted in your browser. This trains a new model; it does not use the old vector-model weights.

L1 penalizes pixel color errors. SSIM penalizes loss of local structure. The combined loss is `alpha*L1 + (1-alpha)*(1-SSIM)`.
The new default latent is **32×16×16 = 8,192 values**, with **zero dropout**. This is 6:1 compression of the input instead of 96:1. No skip connections bypass it.


In [ ]:
from google.colab import drive, files
from pathlib import Path
import os, zipfile, shutil, subprocess, sys, json, sqlite3
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Choose a GPU runtime, then reconnect before training.')
print('GPU:', torch.cuda.get_device_name(0))
drive.mount('/content/drive')
PROJECT = Path('/content/ass1')
BACKUP = Path('/content/drive/MyDrive/GenAI_Ass1/spatial_v2')
PROJECT.mkdir(exist_ok=True)
BACKUP.mkdir(parents=True, exist_ok=True)


In [ ]:
uploaded = files.upload()  # Select colab/task1_source.zip
if len(uploaded) != 1:
    raise ValueError('Upload exactly the generated source ZIP.')
archive_name = next(iter(uploaded))
with zipfile.ZipFile(archive_name) as archive:
    for entry in archive.infolist():
        destination = (PROJECT / entry.filename).resolve()
        if not destination.is_relative_to(PROJECT.resolve()):
            raise ValueError('Invalid ZIP path')
    archive.extractall(PROJECT)
os.chdir(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
# Keep Colab's CUDA-enabled PyTorch installation; do not install the CPU wheel.
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)


In [ ]:
# Restore previous results after reconnecting, and define consistent backup helpers.
for folder in ['runs', 'models', 'mlruns']:
    if (BACKUP / folder).exists():
        shutil.copytree(BACKUP / folder, PROJECT / folder, dirs_exist_ok=True)
if (BACKUP / 'mlflow.db').exists():
    shutil.copy2(BACKUP / 'mlflow.db', PROJECT / 'mlflow.db')

def backup_results():
    ignore = shutil.ignore_patterns('*.sqlite3', '*.db', '*.tmp', '*-journal', '*-wal', '*-shm')
    for folder in ['runs', 'models', 'mlruns']:
        if (PROJECT / folder).exists():
            shutil.copytree(PROJECT / folder, BACKUP / folder, dirs_exist_ok=True, ignore=ignore)
    databases = list((PROJECT / 'runs').rglob('*.sqlite3')) if (PROJECT / 'runs').exists() else []
    if (PROJECT / 'mlflow.db').exists():
        databases.append(PROJECT / 'mlflow.db')
    for source in databases:
        local_copy = Path('/content') / (source.name + '.backup')
        local_copy.unlink(missing_ok=True)
        with sqlite3.connect(str(source), timeout=60) as src, sqlite3.connect(str(local_copy)) as dest:
            src.backup(dest)
        destination = BACKUP / source.relative_to(PROJECT)
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(local_copy, destination)

def run(arguments):
    process = subprocess.Popen([sys.executable, '-u', *arguments], stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, cwd=PROJECT)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue
            if isinstance(record, dict) and 'epoch' in record:
                backup_results()
        if process.wait() != 0:
            raise RuntimeError('Command failed; inspect the output above.')
    except BaseException:
        process.terminate()
        process.wait()
        backup_results()
        raise
    backup_results()


In [ ]:
# Download data to local runtime storage; preserve official split definitions.
import urllib.request, tarfile
raw = PROJECT / 'raw_data'
raw.mkdir(exist_ok=True)
for filename in ['102flowers.tgz', 'setid.mat']:
    path = raw / filename
    if not path.exists():
        temporary = raw / (filename + '.part')
        urllib.request.urlretrieve('https://www.robots.ox.ac.uk/~vgg/data/flowers/102/' + filename, temporary)
        temporary.replace(path)
if not (raw / 'jpg' / 'image_08189.jpg').exists():
    with tarfile.open(raw / '102flowers.tgz') as archive:
        archive.extractall(raw, filter='data')
run(['prepare_dataset.py'])
run(['-m', 'unittest', 'discover', '-s', 'tests', '-v'])


## Tune on validation data
The starting budget is 12 trials × 10 epochs. Change it to match your available GPU time.
Completed trials are retained. After a disconnect, running this cell adds only the remaining
budget, allowing for unfinished trials. The old vector study is stored in a separate directory.


In [ ]:
import optuna
SEARCH = Path('runs/task1/spatial/search')
TRIAL_BUDGET = 12
TRIAL_EPOCHS = 10
study_path = SEARCH / 'study.sqlite3'
finished = 0
if study_path.exists():
    study = optuna.load_study(study_name='task1', storage='sqlite:///' + str(study_path.resolve()))
    finished = sum(t.state in [optuna.trial.TrialState.COMPLETE, optuna.trial.TrialState.PRUNED] for t in study.trials)
remaining = max(0, TRIAL_BUDGET - finished)
if remaining:
    run(['-m', 'task1.train', 'tune', '--config', 'configs/task1.json',
         '--epochs', str(TRIAL_EPOCHS), '--trials', str(remaining), '--device', 'cuda',
         '--workers', '2', '--output', str(SEARCH)])
print(json.loads((SEARCH / 'best_config.json').read_text()))


## Train the selected model
This resumes an interrupted run from the last completed epoch. It preserves the selected
best checkpoint as well as the latest checkpoint. Set the total training budget below.


In [ ]:
FINAL = Path('runs/task1/spatial/final')
TOTAL_EPOCHS = 60
last = FINAL / 'last.pt'
completed = torch.load(last, map_location='cpu', weights_only=True)['epoch'] if last.exists() else 0
if completed < TOTAL_EPOCHS:
    arguments = ['-m', 'task1.train', 'train', '--config', str(SEARCH / 'best_config.json'),
                 '--epochs', str(TOTAL_EPOCHS), '--device', 'cuda', '--workers', '2', '--output', str(FINAL)]
    if last.exists():
        arguments += ['--resume', str(last)]
    run(arguments)
else:
    print('Training already completed:', completed, 'epochs')


In [ ]:
# Inspect validation quality first; do not claim an improvement without these results.
run(['-m', 'task1.evaluate', '--checkpoint', str(FINAL / 'best.pt'), '--split', 'val',
     '--output', 'runs/task1/spatial/validation'])
from IPython.display import display, Image
for panel in sorted(Path('runs/task1/spatial/validation').glob('example_*.png'))[:12]:
    display(Image(filename=str(panel)))


In [ ]:
# Export and verify ONNX parity, then persist the artifacts.
run(['-m', 'task1.export', '--checkpoint', str(FINAL / 'best.pt'),
     '--output', 'models/universal_spatial.onnx'])


## Optional final test comparison
The earlier vector model already used this test set. Any comparison here is a further test-set
measurement, not a fresh unseen evaluation. Keep tuning decisions based on validation.


In [ ]:
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    run(['-m', 'task1.evaluate', '--checkpoint', str(FINAL / 'best.pt'), '--split', 'test',
         '--output', 'runs/task1/spatial/test'])


In [ ]:
backup_results()
# Source images are excluded. Download the trained model and experiment evidence.
output = Path('/content/task1_spatial_results.zip')
with zipfile.ZipFile(output, 'w', zipfile.ZIP_DEFLATED) as archive:
    for folder in ['models', 'runs', 'mlruns']:
        for path in (PROJECT / folder).rglob('*'):
            if path.is_file() and not path.name.endswith(('-journal', '-wal', '-shm', '.tmp')):
                archive.write(path, path.relative_to(PROJECT))
    if (PROJECT / 'mlflow.db').exists():
        archive.write(BACKUP / 'mlflow.db', 'mlflow.db')
files.download(str(output))
